# PROJECT OBJECTIVE

Your final goal is NOT:

descriptive statistics,
or congestion reporting.

The real goal is:

## Identify regions where flexibility/decarbonization services are most needed.

Examples:

* batteries
* peak shaving
* EMS
* smart charging
* load shifting
* industrial flexibility
* renewable integration

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## STEP 1 — LOAD & STANDARDIZE DATA

In [7]:
pc6 = pd.read_csv(
    "raw_data/congestie_pc6.csv",
    sep=";",
    decimal=","
)

vg = pd.read_csv(
    "raw_data/voedingsgebieden.csv",
    sep=";",
    decimal=","
)

tg = pd.read_csv(
    "raw_data/tennetgebieden.csv",
    sep=";",
    decimal=","
)

tc = pd.read_csv(
    "raw_data/tennetcongestie.csv",
    sep=";",
    decimal=","
)

proj = pd.read_csv(
    "raw_data/projecten.csv",
    sep=";",
    decimal=","
)

In [ ]:
pc6 = pc6.rename(columns={ "postcode": "postcode", 
                          "afname": "consumption_congestion", 
                          "opwek": "generation_congestion", 
                          "voedingsgebied_id": "supply_area_id", 
                          "voedingsgebied_naam": "supply_area_name", 
                          "tennet_id": "tennet_id", 
                          "RNB_postcode": "regional_grid_operator" })

vg = vg.rename(columns={ "voedingsgebied_id": "supply_area_id", 
                        "aanwezige_transportcapaciteit_invoeding": "available_feedin_capacity_mw", 
                        "aanwezige_transportcapaciteit_afname": "available_consumption_capacity_mw", 
                        "benodigde_transportcapaciteit_invoeding": "required_feedin_capacity_mw", 
                        "benodigde_transportcapaciteit_afname": "required_consumption_capacity_mw", 
                        "unieke_verzoeken_invoeding": "unique_feedin_requests", 
                        "unieke_verzoeken_afname": "unique_consumption_requests",
                        "wachtrij_invoeding": "feedin_queue_mw", 
                        "wachtrij_afname": "consumption_queue_mw", 
                        "RNB": "regional_grid_operator", 
                        "info": "additional_info" })

tg = tg.rename(columns={ "aanwezige_transportcapaciteit_invoeding": "available_feedin_capacity_mw", 
                        "aanwezige_transportcapaciteit_afname": "available_consumption_capacity_mw", 
                        "benodigde_transportcapaciteit_invoeding": "required_feedin_capacity_mw", 
                        "benodigde_transportcapaciteit_afname": "required_consumption_capacity_mw", 
                        "unieke_verzoeken_invoeding": "unique_feedin_requests", 
                        "unieke_verzoeken_afname": "unique_consumption_requests", 
                        "wachtrij_invoeding": "feedin_queue_mw", 
                        "wachtrij_afname": "consumption_queue_mw", 
                        "congestiegebied": "congestion_area", 
                        "info": "additional_info" })

tc = tc.rename(columns={ "tennet_id": "tennet_id", 
                        "congestiegebied_afname": "consumption_congestion_area", 
                        "congestiegebied_opwek": "generation_congestion_area", 
                        "afname": "consumption_congestion", 
                        "opwek": "generation_congestion" })

proj = proj.rename(columns={ "projectnaam": "project_name", 
                            "gebied_id": "area_id", 
                            "kwartaal": "completion_quarter", 
                            "jaar": "completion_year", 
                            "beschrijving": "description", 
                            "einddatum": "completion_date", 
                            "netbeheerder": "grid_operator" })

In [15]:
print(pc6.shape) 
print(vg.shape) 
print(tg.shape) 
print(tc.shape) 
print(proj.shape)

(462606, 10)
(608, 21)
(150, 23)
(271, 5)
(945, 17)


In [16]:
pc6.isna().sum() 
vg.isna().sum() 
tg.isna().sum() 
tc.isna().sum() 
proj.isna().sum()

Unnamed: 0                 945
project_name                 0
area_id                      0
completion_quarter         525
completion_year              1
description                944
jaar_num                   945
kwartaal_num               945
einddatum_string             0
grid_operator                0
start                      822
eind                       822
tijdschema_beschrijving    273
project_fase               149
bron                         1
project_url                945
project_type               945
dtype: int64

In [22]:
pc6["postcode"].duplicated().sum() 
vg["supply_area_id"].duplicated().sum() 
tc["tennet_id"].duplicated().sum()

np.int64(0)

In [23]:
pc6.info() 
vg.info() 
tg.info() 
tc.info() 
proj.info()

<class 'pandas.DataFrame'>
RangeIndex: 462606 entries, 0 to 462605
Data columns (total 10 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   postcode                462570 non-null  str    
 1   consumption_congestion  462606 non-null  int64  
 2   generation_congestion   462606 non-null  int64  
 3   supply_area_id          462606 non-null  str    
 4   supply_area_name        462570 non-null  str    
 5   tennet_id               462570 non-null  str    
 6   regional_grid_operator  462606 non-null  str    
 7   Gemeentecode            460514 non-null  float64
 8   Gemeentenaam            460514 non-null  str    
 9   provincie               460514 non-null  str    
dtypes: float64(1), int64(2), str(7)
memory usage: 35.3 MB
<class 'pandas.DataFrame'>
RangeIndex: 608 entries, 0 to 607
Data columns (total 21 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                        

In [25]:
proj["completion_date"] = pd.to_datetime(
    proj["einddatum_string"],
    errors="coerce"
)

In [26]:
proj[[
    "einddatum_string",
    "completion_date"
]].head()

,einddatum_string,completion_date
0,2029,2029-01-01
1,2029,2029-01-01
2,2028,2028-01-01
3,2029,2029-01-01
4,2018,2018-01-01


## UNDERSTAND CONGESTION COLORS
Code	Meaning
* -1	No information
* 0	No congestion
* 1	Yellow
* 2	Orange
* 3	Red

In [ ]:
# CONSUMPTION OVERLOAD RATIO

# How much electricity consumers need versus how much the grid can actually transport.

vg["consumption_overload_ratio"] = ( vg["required_consumption_capacity_mw"] / vg["available_consumption_capacity_mw"] )

In [ ]:
# 8.2 GENERATION OVERLOAD RATIO

# The grid cannot absorb all the renewable generation.

vg["generation_overload_ratio"] = (
    vg["required_feedin_capacity_mw"] /
    vg["available_feedin_capacity_mw"]
)

In [ ]:
# 8.3 CONSUMPTION QUEUE PRESSURE



vg["consumption_queue_pressure"] = ( vg["consumption_queue_mw"] / vg["available_consumption_capacity_mw"] )

In [31]:
#8.4 GENERATION QUEUE PRESSURE

vg["generation_queue_pressure"] = (
    vg["feedin_queue_mw"] /
    vg["available_feedin_capacity_mw"]
)

In [32]:
# 8.5 TOTAL REQUEST PRESSURE

vg["total_requests"] = (
    vg["unique_feedin_requests"] +
    vg["unique_consumption_requests"]
)

## CREATE OPPORTUNITY SCORES

### 9.1 FLEXIBILITY OPPORTUNITY SCORE

Focus:

* batteries,
* EMS,
* peak shaving,
* demand response.

In [33]:
vg["flexibility_score"] = ( 
                           0.35 * vg["consumption_overload_ratio"] + 
                           0.30 * vg["consumption_queue_pressure"] + 
                           0.20 * vg["unique_consumption_requests"] + 
                           0.15 * ( vg["consumption_congestion_proxy"] if "consumption_congestion_proxy" in vg.columns else 0 ) )

### 9.2 STORAGE OPPORTUNITY SCORE

Focus:

* battery storage,
* renewable balancing,
* curtailment optimization.

In [34]:
vg["storage_score"] = ( 
                       0.40 * vg["generation_overload_ratio"] + 
                       0.30 * vg["generation_queue_pressure"] + 
                       0.30 * vg["unique_feedin_requests"] )

## 10. CLASSIFY REGIONS


In [35]:
# 10.1 CREATE REGION TYPES
def classify_region(row):

    if (
        row["consumption_overload_ratio"] > 1.1 and
        row["generation_overload_ratio"] > 1.1
    ):
        return "mixed_congestion"

    elif row["consumption_overload_ratio"] > 1.1:
        return "electrification_bottleneck"

    elif row["generation_overload_ratio"] > 1.1:
        return "renewable_bottleneck"

    else:
        return "normal"


vg["region_type"] = vg.apply(
    classify_region,
    axis=1
)

## 11. MERGE DATASETS

In [ ]:
# 11.1 POSTCODE + SUPPLY AREA
pc6_vg = pc6.merge(
    vg,
    on="supply_area_id",
    how="left"
)

In [37]:
pc6_vg

,postcode,consumption_congestion,generation_congestion,supply_area_id,supply_area_name,tennet_id,regional_grid_operator_x,Gemeentecode,Gemeentenaam,provincie_x,...,regional_grid_operator_y,provincie_y,consumption_overload_ratio,generation_overload_ratio,consumption_queue_pressure,generation_queue_pressure,total_requests,flexibility_score,storage_score,region_type
0,7471AA,1,1,GO,GO - Goor,Station Goor 110 kV,Coteq,1735.0,Hof van Twente,Overijssel,...,"Coteq, Enexis",Overijssel,0.583285,0.738394,0.054169,0.060734,47.0,6.220400,5.413578,normal
1,7471AB,1,1,GO,GO - Goor,Station Goor 110 kV,Coteq,1735.0,Hof van Twente,Overijssel,...,"Coteq, Enexis",Overijssel,0.583285,0.738394,0.054169,0.060734,47.0,6.220400,5.413578,normal
2,7471AC,1,1,GO,GO - Goor,Station Goor 110 kV,Coteq,1735.0,Hof van Twente,Overijssel,...,"Coteq, Enexis",Overijssel,0.583285,0.738394,0.054169,0.060734,47.0,6.220400,5.413578,normal
3,7471AD,1,1,GO,GO - Goor,Station Goor 110 kV,Coteq,1735.0,Hof van Twente,Overijssel,...,"Coteq, Enexis",Overijssel,0.583285,0.738394,0.054169,0.060734,47.0,6.220400,5.413578,normal
4,7471AE,1,1,GO,GO - Goor,Station Goor 110 kV,Coteq,1735.0,Hof van Twente,Overijssel,...,"Coteq, Enexis",Overijssel,0.583285,0.738394,0.054169,0.060734,47.0,6.220400,5.413578,normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
462601,2291VG,0,2,Westerlee,Westerlee,Station Westerlee 150 kV,Westland,1783.0,Westland,Zuid-Holland,...,Westland,Zuid-Holland,0.634921,1.000000,0.000000,0.275132,171.0,0.222222,51.782540,normal
462602,2675WC,0,2,Westerlee,Westerlee,Station Westerlee 150 kV,Westland,1783.0,Westland,Zuid-Holland,...,Westland,Zuid-Holland,0.634921,1.000000,0.000000,0.275132,171.0,0.222222,51.782540,normal
462603,2681SW,0,2,Westerlee,Westerlee,Station Westerlee 150 kV,Westland,1783.0,Westland,Zuid-Holland,...,Westland,Zuid-Holland,0.634921,1.000000,0.000000,0.275132,171.0,0.222222,51.782540,normal
462604,2635GN,0,0,De Lier,De Lier,Station De Lier 150 kV,Westland,1842.0,Midden-Delfland,Zuid-Holland,...,Westland,Zuid-Holland,0.330159,0.761905,0.000000,0.000000,0.0,0.115556,0.304762,normal


In [38]:
pc6_full = pc6_vg.merge( tc, on="tennet_id", how="left", suffixes=("", "_tennet") )

In [75]:
pc6_full

,postcode,consumption_congestion,generation_congestion,supply_area_id,supply_area_name,tennet_id,regional_grid_operator_x,Gemeentecode,Gemeentenaam,provincie_x,...,consumption_queue_pressure,generation_queue_pressure,total_requests,flexibility_score,storage_score,region_type,consumption_congestion_area,generation_congestion_area,consumption_congestion_tennet,generation_congestion_tennet
0,7471AA,1,1,GO,GO - Goor,Station Goor 110 kV,Coteq,1735.0,Hof van Twente,Overijssel,...,0.054169,0.060734,47.0,6.220400,5.413578,normal,Overijssel,Hengelo,3.0,3.0
1,7471AB,1,1,GO,GO - Goor,Station Goor 110 kV,Coteq,1735.0,Hof van Twente,Overijssel,...,0.054169,0.060734,47.0,6.220400,5.413578,normal,Overijssel,Hengelo,3.0,3.0
2,7471AC,1,1,GO,GO - Goor,Station Goor 110 kV,Coteq,1735.0,Hof van Twente,Overijssel,...,0.054169,0.060734,47.0,6.220400,5.413578,normal,Overijssel,Hengelo,3.0,3.0
3,7471AD,1,1,GO,GO - Goor,Station Goor 110 kV,Coteq,1735.0,Hof van Twente,Overijssel,...,0.054169,0.060734,47.0,6.220400,5.413578,normal,Overijssel,Hengelo,3.0,3.0
4,7471AE,1,1,GO,GO - Goor,Station Goor 110 kV,Coteq,1735.0,Hof van Twente,Overijssel,...,0.054169,0.060734,47.0,6.220400,5.413578,normal,Overijssel,Hengelo,3.0,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
462601,2291VG,0,2,Westerlee,Westerlee,Station Westerlee 150 kV,Westland,1783.0,Westland,Zuid-Holland,...,0.000000,0.275132,171.0,0.222222,51.782540,normal,Westland,Westland,0.0,0.0
462602,2675WC,0,2,Westerlee,Westerlee,Station Westerlee 150 kV,Westland,1783.0,Westland,Zuid-Holland,...,0.000000,0.275132,171.0,0.222222,51.782540,normal,Westland,Westland,0.0,0.0
462603,2681SW,0,2,Westerlee,Westerlee,Station Westerlee 150 kV,Westland,1783.0,Westland,Zuid-Holland,...,0.000000,0.275132,171.0,0.222222,51.782540,normal,Westland,Westland,0.0,0.0
462604,2635GN,0,0,De Lier,De Lier,Station De Lier 150 kV,Westland,1842.0,Midden-Delfland,Zuid-Holland,...,0.000000,0.000000,0.0,0.115556,0.304762,normal,Westland,Westland,0.0,0.0


## TERRITORIAL ANALYSIS

In [41]:
# TOP FLEXIBILITY REGIONS

vg.sort_values( "flexibility_score", ascending=False )[[ "supply_area_id", "regional_grid_operator", "flexibility_score" ]].head(20)

,supply_area_id,regional_grid_operator,flexibility_score
142,Medemblik Transformator 1,Liander,inf
121,HTN,Enexis,56.040420
6,BD,Enexis,37.794092
94,ERD,Enexis,36.907631
391,RSD,Enexis,35.664849
117,HMZ,Enexis,34.530347
589,VENR,Enexis,33.876298
601,WW,Enexis,32.538146
587,UD,Enexis,31.271095
398,TBN,Enexis,31.076417


In [42]:
# 12.2 TOP STORAGE REGIONS


vg.sort_values(
    "storage_score",
    ascending=False
)[[
    "supply_area_id",
    "storage_score"
]].head(20)

,supply_area_id,storage_score
602,Westerlee,51.782540
589,VENR,39.139532
601,WW,37.581419
117,HMZ,35.475838
587,UD,34.792724
119,HPT,34.729345
121,HTN,33.273681
94,ERD,33.124944
99,ETN,32.409047
120,HRST,31.716822


In [43]:
# 12.3 MOST STRESSED OPERATORS

vg.groupby( "regional_grid_operator" )[[ "flexibility_score", "storage_score" ]].mean()

,flexibility_score,storage_score
regional_grid_operator,,
"Coteq, Enexis",7.227990,10.112501
Enexis,12.517439,12.281082
"Enexis, Rendo",6.489580,8.426302
Liander,inf,7.206278
Stedin,3.824530,3.650456
Westland,0.168889,26.043651


In [ ]:
# 12.4 MOST CONGESTED POSTCODES

pc6[
    pc6["consumption_congestion"] == 3
]

,postcode,consumption_congestion,generation_congestion,supply_area_id,supply_area_name,tennet_id,regional_grid_operator,Gemeentecode,Gemeentenaam,provincie
1256,7601AA,3,3,AMLU,AMLU - Almelo Urenco,Station Almelo Urenco 110 kV,Coteq,141.0,Almelo,Overijssel
1257,7601AB,3,3,AMLU,AMLU - Almelo Urenco,Station Almelo Urenco 110 kV,Coteq,141.0,Almelo,Overijssel
1258,7601AC,3,3,AMLU,AMLU - Almelo Urenco,Station Almelo Urenco 110 kV,Coteq,141.0,Almelo,Overijssel
1259,7601AD,3,3,AMLU,AMLU - Almelo Urenco,Station Almelo Urenco 110 kV,Coteq,141.0,Almelo,Overijssel
1260,7601AE,3,3,AMLU,AMLU - Almelo Urenco,Station Almelo Urenco 110 kV,Coteq,141.0,Almelo,Overijssel
...,...,...,...,...,...,...,...,...,...,...
459250,2555LP,3,0,CM49,Appelstraat,Station Rijswijk 150 kV,Stedin,518.0,'s-Gravenhage,Zuid-Holland
459251,2522HA,3,1,CM52,Jan Wapstraat,Station Rijswijk 150 kV,Stedin,518.0,'s-Gravenhage,Zuid-Holland
459252,2555LS,3,0,CM49,Appelstraat,Station Rijswijk 150 kV,Stedin,518.0,'s-Gravenhage,Zuid-Holland
459253,2935BP,3,1,CM48,Langeland,Station Krimpen a/d IJssel 150 kV,Stedin,1931.0,Krimpenerwaard,Zuid-Holland


## 13. PROJECT PIPELINE ANALYSIS

Projects indicate:

whether congestion is temporary,
or structural.

In [49]:
# 13.1 YEARS UNTIL COMPLETION

proj["completion_year"] = pd.to_numeric(
    proj["completion_year"],
    errors="coerce"
)

current_year = 2026

proj["years_until_completion"] = (
    proj["completion_year"] -
    current_year
)

In [50]:
# 13.2 LONG-DELAY PROJECTS
proj[
    proj["years_until_completion"] >= 5
]


,Unnamed: 0,project_name,area_id,completion_quarter,completion_year,description,jaar_num,kwartaal_num,einddatum_string,grid_operator,start,eind,tijdschema_beschrijving,project_fase,bron,project_url,project_type,completion_date,years_until_completion
6,NaN,Extra capaciteit op Uden,UD,NaN,2031.0,NaN,NaN,NaN,2031,Enexis,NaN,NaN,NaN,scheduled,Laatste inzichten,NaN,NaN,2031-01-01,5.0
13,NaN,Extra capaciteit op Boekend,BOEK,NaN,2033.0,NaN,NaN,NaN,2033,Enexis,NaN,NaN,NaN,scheduled,Laatste inzichten,NaN,NaN,2033-01-01,7.0
18,NaN,Extra capaciteit op Oosteind,OTD,NaN,2031.0,NaN,NaN,NaN,2031,Enexis,NaN,NaN,NaN,scheduled,Laatste inzichten,NaN,NaN,2031-01-01,5.0
19,NaN,Extra capaciteit op Blerick,BLER,NaN,2031.0,NaN,NaN,NaN,2031,Enexis,NaN,NaN,NaN,scheduled,Laatste inzichten,NaN,NaN,2031-01-01,5.0
22,NaN,Extra capaciteit op Helmond Oost,HMO,NaN,2032.0,NaN,NaN,NaN,2032,Enexis,NaN,NaN,NaN,scheduled,Laatste inzichten,NaN,NaN,2032-01-01,6.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
940,NaN,Borssele-Rilland 380 kV verzwaren verbinding,Zeeland,4.0,2037.0,NaN,NaN,NaN,2037 - Q4,TenneT,2033-01-01 00:00:00,2037-12-31,NaN,approved,Investeringsplan,NaN,NaN,NaT,11.0
941,NaN,Terneuzen 380 kV realiseren station,Zeeland,4.0,2037.0,NaN,NaN,NaN,2037 - Q4,TenneT,2034-01-01 00:00:00,2037-12-31,NaN,scheduled,Investeringsplan,NaN,NaN,NaT,11.0
942,NaN,Borssele-Goes de Poel-Terneuzen 150 kV aanpass...,Zeeland,4.0,2038.0,NaN,NaN,NaN,2038 - Q4,TenneT,2035-01-01 00:00:00,2038-12-31,NaN,approved,Investeringsplan,NaN,NaN,NaT,12.0
943,NaN,Terneuzen-Westdorpe 150 kV verzwaren verbinding,Zeeland,4.0,2038.0,NaN,NaN,NaN,2038 - Q4,TenneT,2035-01-01 00:00:00,2038-12-31,NaN,approved,Investeringsplan,NaN,NaN,NaT,12.0


## CREATE FINAL STRATEGIC SEGMENTS

### Segment 1 — Electrification Bottlenecks

#### Conditions:

* high consumption overload,
* high demand queues.

#### Best services:

* peak shaving,
* EMS,
* smart charging.


### Segment 2 — Renewable Bottlenecks

#### Conditions:

* high generation overload,
* large feed-in queues.

#### Best services:

* battery storage,
* renewable optimization.


### Segment 3 — Structural Congestion

#### Conditions:

* both overloaded.

#### Best services:

* advanced flexibility platforms,
* local energy systems.

In [51]:
vg["consumption_overload_ratio"] = (
    vg["required_consumption_capacity_mw"] /
    vg["available_consumption_capacity_mw"]
)

vg["generation_overload_ratio"] = (
    vg["required_feedin_capacity_mw"] /
    vg["available_feedin_capacity_mw"]
)

vg["consumption_queue_pressure"] = (
    vg["consumption_queue_mw"] /
    vg["available_consumption_capacity_mw"]
)

vg["generation_queue_pressure"] = (
    vg["feedin_queue_mw"] /
    vg["available_feedin_capacity_mw"]
)

vg["total_requests"] = (
    vg["unique_feedin_requests"] +
    vg["unique_consumption_requests"]
)

In [52]:
def assign_segment(row):

    consumption_stress = (
        row["consumption_overload_ratio"] > 1.1
        or row["consumption_queue_pressure"] > 0.2
    )

    generation_stress = (
        row["generation_overload_ratio"] > 1.1
        or row["generation_queue_pressure"] > 0.2
    )

    # Segment 1 — Electrification bottleneck
    if consumption_stress and not generation_stress:
        return "electrification_bottleneck"

    # Segment 2 — Renewable bottleneck
    elif generation_stress and not consumption_stress:
        return "renewable_bottleneck"

    # Segment 3 — Structural congestion
    elif consumption_stress and generation_stress:
        return "structural_congestion"

    else:
        return "normal"

In [54]:
vg["segment"] = vg.apply(assign_segment, axis=1)

In [55]:
vg

,supply_area_id,jaar,available_feedin_capacity_mw,available_consumption_capacity_mw,required_feedin_capacity_mw,required_consumption_capacity_mw,unique_feedin_requests,unique_consumption_requests,feedin_queue_mw,consumption_queue_mw,...,provincie,consumption_overload_ratio,generation_overload_ratio,consumption_queue_pressure,generation_queue_pressure,total_requests,flexibility_score,storage_score,region_type,segment
0,118065277_23,2026,NaN,NaN,NaN,NaN,0.0,0.0,0.000,0.00000,...,Zuid-Holland,NaN,NaN,NaN,NaN,0.0,NaN,NaN,normal,normal
1,AMLM,2026,78.49,83.56,24.714790,79.358827,44.0,42.0,39.418,10.55500,...,Overijssel,0.949723,0.314878,0.126316,0.502204,86.0,8.770298,13.476613,normal,renewable_bottleneck
2,AMLU,2026,62.43,65.55,26.589619,35.289319,35.0,18.0,36.056,18.51400,...,Overijssel,0.538357,0.425911,0.282441,0.577543,53.0,3.873157,10.843627,normal,structural_congestion
3,ARI,2026,98.51,108.77,112.935959,73.623838,61.0,89.0,70.009,49.32100,...,Noord-Brabant,0.676876,1.146442,0.453443,0.710679,150.0,18.172940,18.971780,renewable_bottleneck,structural_congestion
4,Amstelveen,2026,63.00,56.00,16.000000,53.000000,NaN,23.0,NaN,5.17136,...,Noord-Holland,0.946429,0.253968,0.092346,NaN,NaN,4.958954,NaN,normal,normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
603,ZLF,2026,89.55,89.55,33.621028,55.059499,27.0,45.0,10.860,24.15500,...,Overijssel,0.614846,0.375444,0.269738,0.121273,72.0,9.296118,8.286560,normal,electrification_bottleneck
604,ZLH,2026,90.73,90.73,80.280949,70.256925,32.0,41.0,96.695,45.92100,...,Overijssel,0.774352,0.884834,0.506128,1.065745,73.0,8.622861,10.273657,normal,structural_congestion
605,ZLW,2026,84.99,88.19,40.058644,66.604687,51.0,79.0,29.301,28.40300,...,Overijssel,0.755241,0.471334,0.322066,0.344758,130.0,16.160954,15.591961,normal,structural_congestion
606,ZS,2026,62.13,65.48,24.778826,46.210547,35.0,26.0,18.043,9.54600,...,"Drenthe, Overijssel",0.705720,0.398822,0.145785,0.290407,61.0,5.490737,10.746651,normal,renewable_bottleneck


In [56]:
vg["queue_pressure"] = (
    vg["consumption_queue_pressure"] +
    vg["generation_queue_pressure"]
)

In [57]:
vg["flexibility_score"] = (
    0.30 * vg["consumption_overload_ratio"].fillna(0) +
    0.25 * vg["consumption_queue_pressure"].fillna(0) +
    0.15 * vg["generation_queue_pressure"].fillna(0) +
    0.20 * vg["unique_consumption_requests"].fillna(0) +
    0.10 * vg["total_requests"].fillna(0)
)

In [58]:
vg["storage_score"] = (
    0.35 * vg["generation_overload_ratio"].fillna(0) +
    0.25 * vg["generation_queue_pressure"].fillna(0) +
    0.20 * vg["unique_feedin_requests"].fillna(0) +
    0.20 * vg["total_requests"].fillna(0)
)

In [59]:
vg["system_stress_index"] = (
    vg["consumption_overload_ratio"].fillna(0) +
    vg["generation_overload_ratio"].fillna(0) +
    vg["consumption_queue_pressure"].fillna(0) +
    vg["generation_queue_pressure"].fillna(0)
)

In [60]:
vg

,supply_area_id,jaar,available_feedin_capacity_mw,available_consumption_capacity_mw,required_feedin_capacity_mw,required_consumption_capacity_mw,unique_feedin_requests,unique_consumption_requests,feedin_queue_mw,consumption_queue_mw,...,generation_overload_ratio,consumption_queue_pressure,generation_queue_pressure,total_requests,flexibility_score,storage_score,region_type,segment,queue_pressure,system_stress_index
0,118065277_23,2026,NaN,NaN,NaN,NaN,0.0,0.0,0.000,0.00000,...,NaN,NaN,NaN,0.0,0.000000,0.000000,normal,normal,NaN,0.000000
1,AMLM,2026,78.49,83.56,24.714790,79.358827,44.0,42.0,39.418,10.55500,...,0.314878,0.126316,0.502204,86.0,17.391827,26.235758,normal,renewable_bottleneck,0.628521,1.893121
2,AMLU,2026,62.43,65.55,26.589619,35.289319,35.0,18.0,36.056,18.51400,...,0.425911,0.282441,0.577543,53.0,9.218749,17.893455,normal,structural_congestion,0.859984,1.824252
3,ARI,2026,98.51,108.77,112.935959,73.623838,61.0,89.0,70.009,49.32100,...,1.146442,0.453443,0.710679,150.0,33.223026,42.778924,renewable_bottleneck,structural_congestion,1.164122,2.987440
4,Amstelveen,2026,63.00,56.00,16.000000,53.000000,NaN,23.0,NaN,5.17136,...,0.253968,0.092346,NaN,NaN,4.907015,0.088889,normal,normal,NaN,1.292743
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
603,ZLF,2026,89.55,89.55,33.621028,55.059499,27.0,45.0,10.860,24.15500,...,0.375444,0.269738,0.121273,72.0,16.470079,19.961724,normal,electrification_bottleneck,0.391011,1.381301
604,ZLH,2026,90.73,90.73,80.280949,70.256925,32.0,41.0,96.695,45.92100,...,0.884834,0.506128,1.065745,73.0,16.018699,21.576128,normal,structural_congestion,1.571873,3.231058
605,ZLW,2026,84.99,88.19,40.058644,66.604687,51.0,79.0,29.301,28.40300,...,0.471334,0.322066,0.344758,130.0,29.158802,36.451156,normal,structural_congestion,0.666824,1.893399
606,ZS,2026,62.13,65.48,24.778826,46.210547,35.0,26.0,18.043,9.54600,...,0.398822,0.145785,0.290407,61.0,11.591723,19.412190,normal,renewable_bottleneck,0.436192,1.540734


## 16. BUSINESS INTERPRETATION → TURN INTO DECISION RULES

This is where your analysis becomes a product.

### RULE 1 — WHERE TO SELL PEAK SHAVING
IF consumption_overload_ratio > 1.1
AND consumption_queue_pressure high
### → SELL flexibility / EMS / peak shaving

In [76]:
peak_shaving_targets = vg[
    vg["consumption_overload_ratio"] > 1.1
].sort_values("flexibility_score", ascending=False)

peak_shaving_targets

,supply_area_id,jaar,available_feedin_capacity_mw,available_consumption_capacity_mw,required_feedin_capacity_mw,required_consumption_capacity_mw,unique_feedin_requests,unique_consumption_requests,feedin_queue_mw,consumption_queue_mw,...,consumption_queue_pressure,generation_queue_pressure,total_requests,flexibility_score,storage_score,region_type,segment,queue_pressure,system_stress_index,recommended_service
142,Medemblik Transformator 1,2026,40.00,0.00,14.000000,1.000000,NaN,7.0,NaN,1.62892,...,inf,NaN,NaN,inf,0.122500,electrification_bottleneck,electrification_bottleneck,NaN,inf,peak_shaving + EMS + smart_charging
347,OS WEESP 10-9i,2026,20.00,0.00,5.000000,10.000000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,inf,0.087500,electrification_bottleneck,electrification_bottleneck,NaN,inf,peak_shaving + EMS + smart_charging
177,OS DALE 10-1i,2026,76.00,57.00,61.000000,68.000000,41.0,66.0,10.704,10.39350,...,0.182342,0.140842,107.0,24.324607,29.916132,electrification_bottleneck,electrification_bottleneck,0.323184,2.318798,peak_shaving + EMS + smart_charging
286,OS OUDORP 10-1i,2026,36.00,36.00,26.000000,40.000000,20.0,63.0,12.102,28.32516,...,0.786810,0.336167,83.0,21.480461,20.936819,electrification_bottleneck,structural_congestion,1.122977,2.956310,full_flex_platform + local_energy_systems
323,OS TIEL 10-1i,2026,77.00,37.00,41.000000,41.000000,48.0,53.0,22.236,19.06852,...,0.515365,0.288779,101.0,21.204591,30.058558,electrification_bottleneck,structural_congestion,0.804145,2.444720,full_flex_platform + local_energy_systems
154,OS ALMERE 10-1i,2026,104.00,59.00,30.000000,67.000000,34.0,39.0,7.935,8.21890,...,0.139303,0.076298,73.0,15.486949,21.520036,electrification_bottleneck,electrification_bottleneck,0.215601,1.639656,peak_shaving + EMS + smart_charging
151,OS AALSMEER BLOEMENVEILING 10-1i,2026,36.00,36.00,20.000000,51.000000,8.0,38.0,4.306,22.99782,...,0.638828,0.119611,46.0,12.802649,11.024347,electrification_bottleneck,electrification_bottleneck,0.758439,2.730662,peak_shaving + EMS + smart_charging
105,GNHK,2026,86.39,86.39,10.653840,96.224143,22.0,31.0,6.060,59.47300,...,0.688425,0.070147,53.0,12.016778,15.060700,electrification_bottleneck,electrification_bottleneck,0.758572,1.995728,peak_shaving + EMS + smart_charging
5,BBS,2026,21.60,21.60,9.031061,24.608277,15.0,27.0,6.632,5.88000,...,0.272222,0.307037,42.0,10.055893,11.623096,electrification_bottleneck,structural_congestion,0.579259,2.136636,full_flex_platform + local_energy_systems
229,OS HOOFDDORP 10-2i,2026,32.00,19.00,8.000000,27.000000,NaN,46.0,NaN,12.95876,...,0.682040,NaN,NaN,9.796826,0.087500,electrification_bottleneck,electrification_bottleneck,NaN,2.353093,peak_shaving + EMS + smart_charging


In [62]:
battery_targets = vg[
    vg["generation_overload_ratio"] > 1.1
].sort_values("storage_score", ascending=False)

In [64]:
structural_targets = vg[
    vg["segment"] == "structural_congestion"
].sort_values("system_stress_index", ascending=False)

structural_targets

,supply_area_id,jaar,available_feedin_capacity_mw,available_consumption_capacity_mw,required_feedin_capacity_mw,required_consumption_capacity_mw,unique_feedin_requests,unique_consumption_requests,feedin_queue_mw,consumption_queue_mw,...,generation_overload_ratio,consumption_queue_pressure,generation_queue_pressure,total_requests,flexibility_score,storage_score,region_type,segment,queue_pressure,system_stress_index
141,MZ,2026,81.89,86.39,110.618767,48.120283,38.0,96.0,124.612,64.03400,...,1.350821,0.741220,1.521700,134.0,33.180664,35.253212,renewable_bottleneck,structural_congestion,2.262920,4.170754
330,OS ULFT 20-3i,2026,46.00,46.00,68.000000,13.000000,14.0,18.0,83.178,22.29860,...,1.478261,0.484752,1.808217,32.0,7.277203,10.169446,renewable_bottleneck,structural_congestion,2.292970,4.053839
166,OS BERGUM CENTRALE 10-2i,2026,17.00,17.00,24.000000,32.000000,30.0,19.0,5.173,3.81204,...,1.411765,0.224238,0.304294,49.0,9.366409,16.370191,mixed_congestion,structural_congestion,0.528532,3.822649
157,OS ALPHEN WEST 10-1i,2026,49.00,42.00,34.000000,43.000000,12.0,31.0,24.889,57.55908,...,0.693878,1.370454,0.507939,43.0,11.225947,11.369842,normal,structural_congestion,1.878393,3.596080
593,VO,2026,23.94,23.94,28.790280,20.784332,22.0,25.0,20.590,12.22900,...,1.202601,0.510819,0.860067,47.0,10.217170,14.435927,renewable_bottleneck,structural_congestion,1.370886,3.441671
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119,HPT,2026,186.90,197.90,74.487476,79.047444,115.0,141.0,43.565,54.87600,...,0.398542,0.277292,0.233093,256.0,54.024116,74.397763,normal,structural_congestion,0.510384,1.308357
96,ESDM,2026,41.80,41.80,5.058799,20.881787,22.0,28.0,10.600,16.47800,...,0.121024,0.394211,0.253589,50.0,10.886460,14.505755,normal,structural_congestion,0.647799,1.268387
15,BOZ,2026,175.06,177.96,36.328090,76.450317,45.0,71.0,44.680,63.23200,...,0.207518,0.355316,0.255227,116.0,26.055991,32.336438,normal,structural_congestion,0.610543,1.247653
367,OS ZALTBOMMEL 20-5i,2026,145.00,94.00,30.000000,25.000000,41.0,34.0,54.045,34.49030,...,0.206897,0.366918,0.372724,75.0,14.527425,23.365595,normal,structural_congestion,0.739642,1.212496


In [65]:
opportunity_ranking = vg[[
    "supply_area_id",
    "regional_grid_operator",
    "segment",
    "flexibility_score",
    "storage_score",
    "system_stress_index"
]].sort_values(
    "system_stress_index",
    ascending=False
)

opportunity_ranking

,supply_area_id,regional_grid_operator,segment,flexibility_score,storage_score,system_stress_index
142,Medemblik Transformator 1,Liander,electrification_bottleneck,inf,0.122500,inf
347,OS WEESP 10-9i,Liander,electrification_bottleneck,inf,0.087500,inf
141,MZ,Enexis,structural_congestion,33.180664,35.253212,4.170754
330,OS ULFT 20-3i,Liander,structural_congestion,7.277203,10.169446,4.053839
166,OS BERGUM CENTRALE 10-2i,Liander,structural_congestion,9.366409,16.370191,3.822649
...,...,...,...,...,...,...
578,TT211_10,Stedin,normal,0.000000,0.000000,0.000000
569,TT115_10,Stedin,normal,0.000000,0.000000,0.000000
570,TT119_23,Stedin,normal,0.000000,0.000000,0.000000
592,VLH DE VOORST 61 10-1i,Liander,normal,0.000000,0.000000,0.000000


In [67]:
map_data = vg[[
    "supply_area_id",
    "segment",
    "flexibility_score",
    "storage_score",
    "system_stress_index"
]]

map_data

,supply_area_id,segment,flexibility_score,storage_score,system_stress_index
0,118065277_23,normal,0.000000,0.000000,0.000000
1,AMLM,renewable_bottleneck,17.391827,26.235758,1.893121
2,AMLU,structural_congestion,9.218749,17.893455,1.824252
3,ARI,structural_congestion,33.223026,42.778924,2.987440
4,Amstelveen,normal,4.907015,0.088889,1.292743
...,...,...,...,...,...
603,ZLF,electrification_bottleneck,16.470079,19.961724,1.381301
604,ZLH,structural_congestion,16.018699,21.576128,3.231058
605,ZLW,structural_congestion,29.158802,36.451156,1.893399
606,ZS,renewable_bottleneck,11.591723,19.412190,1.540734


In [69]:
def recommend_service(row):

    if row["segment"] == "electrification_bottleneck":
        return "peak_shaving + EMS + smart_charging"

    elif row["segment"] == "renewable_bottleneck":
        return "battery_storage + curtailment_management"

    elif row["segment"] == "structural_congestion":
        return "full_flex_platform + local_energy_systems"

    else:
        return "monitor"

vg["recommended_service"] = vg.apply(recommend_service, axis=1)
vg

,supply_area_id,jaar,available_feedin_capacity_mw,available_consumption_capacity_mw,required_feedin_capacity_mw,required_consumption_capacity_mw,unique_feedin_requests,unique_consumption_requests,feedin_queue_mw,consumption_queue_mw,...,consumption_queue_pressure,generation_queue_pressure,total_requests,flexibility_score,storage_score,region_type,segment,queue_pressure,system_stress_index,recommended_service
0,118065277_23,2026,NaN,NaN,NaN,NaN,0.0,0.0,0.000,0.00000,...,NaN,NaN,0.0,0.000000,0.000000,normal,normal,NaN,0.000000,monitor
1,AMLM,2026,78.49,83.56,24.714790,79.358827,44.0,42.0,39.418,10.55500,...,0.126316,0.502204,86.0,17.391827,26.235758,normal,renewable_bottleneck,0.628521,1.893121,battery_storage + curtailment_management
2,AMLU,2026,62.43,65.55,26.589619,35.289319,35.0,18.0,36.056,18.51400,...,0.282441,0.577543,53.0,9.218749,17.893455,normal,structural_congestion,0.859984,1.824252,full_flex_platform + local_energy_systems
3,ARI,2026,98.51,108.77,112.935959,73.623838,61.0,89.0,70.009,49.32100,...,0.453443,0.710679,150.0,33.223026,42.778924,renewable_bottleneck,structural_congestion,1.164122,2.987440,full_flex_platform + local_energy_systems
4,Amstelveen,2026,63.00,56.00,16.000000,53.000000,NaN,23.0,NaN,5.17136,...,0.092346,NaN,NaN,4.907015,0.088889,normal,normal,NaN,1.292743,monitor
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
603,ZLF,2026,89.55,89.55,33.621028,55.059499,27.0,45.0,10.860,24.15500,...,0.269738,0.121273,72.0,16.470079,19.961724,normal,electrification_bottleneck,0.391011,1.381301,peak_shaving + EMS + smart_charging
604,ZLH,2026,90.73,90.73,80.280949,70.256925,32.0,41.0,96.695,45.92100,...,0.506128,1.065745,73.0,16.018699,21.576128,normal,structural_congestion,1.571873,3.231058,full_flex_platform + local_energy_systems
605,ZLW,2026,84.99,88.19,40.058644,66.604687,51.0,79.0,29.301,28.40300,...,0.322066,0.344758,130.0,29.158802,36.451156,normal,structural_congestion,0.666824,1.893399,full_flex_platform + local_energy_systems
606,ZS,2026,62.13,65.48,24.778826,46.210547,35.0,26.0,18.043,9.54600,...,0.145785,0.290407,61.0,11.591723,19.412190,normal,renewable_bottleneck,0.436192,1.540734,battery_storage + curtailment_management


In [70]:
proj["completion_year"] = pd.to_numeric(
    proj["completion_year"],
    errors="coerce"
)

In [71]:
proj["delay_score"] = 2030 - proj["completion_year"]

In [72]:
def delay_category(x):

    if x >= 5:
        return "long_term_gap"

    elif x >= 2:
        return "mid_term_gap"

    else:
        return "short_term_fix"


proj["delay_category"] = proj["delay_score"].apply(delay_category)

In [73]:
vg_projects = vg.merge(
    proj,
    left_on="supply_area_id",
    right_on="area_id",
    how="left"
)

In [74]:
vg_projects

,supply_area_id,jaar,available_feedin_capacity_mw,available_consumption_capacity_mw,required_feedin_capacity_mw,required_consumption_capacity_mw,unique_feedin_requests,unique_consumption_requests,feedin_queue_mw,consumption_queue_mw,...,eind_y,tijdschema_beschrijving,project_fase,bron,project_url,project_type,completion_date,years_until_completion,delay_score,delay_category
0,118065277_23,2026,NaN,NaN,NaN,NaN,0.0,0.0,0.000,0.00000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN
1,AMLM,2026,78.49,83.56,24.714790,79.358827,44.0,42.0,39.418,10.55500,...,NaN,NaN,inProgress,Laatste inzichten,NaN,NaN,2029-01-01,3.0,1.0,short_term_fix
2,AMLU,2026,62.43,65.55,26.589619,35.289319,35.0,18.0,36.056,18.51400,...,NaN,NaN,scheduled,Laatste inzichten,NaN,NaN,2028-01-01,2.0,2.0,mid_term_gap
3,ARI,2026,98.51,108.77,112.935959,73.623838,61.0,89.0,70.009,49.32100,...,NaN,NaN,scheduled,Laatste inzichten,NaN,NaN,2031-01-01,5.0,-1.0,short_term_fix
4,Amstelveen,2026,63.00,56.00,16.000000,53.000000,NaN,23.0,NaN,5.17136,...,NaN,Eerste inschatting,NaN,Laatste Inzichten,NaN,NaN,NaT,6.0,-2.0,short_term_fix
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1061,ZLW,2026,84.99,88.19,40.058644,66.604687,51.0,79.0,29.301,28.40300,...,NaN,NaN,scheduled,Laatste inzichten,NaN,NaN,2031-01-01,5.0,-1.0,short_term_fix
1062,ZLW,2026,84.99,88.19,40.058644,66.604687,51.0,79.0,29.301,28.40300,...,NaN,NaN,approved,Laatste inzichten,NaN,NaN,2030-01-01,4.0,0.0,short_term_fix
1063,ZS,2026,62.13,65.48,24.778826,46.210547,35.0,26.0,18.043,9.54600,...,NaN,NaN,scheduled,Laatste inzichten,NaN,NaN,2029-01-01,3.0,1.0,short_term_fix
1064,ZYV,2026,206.99,208.37,103.980452,68.683093,19.0,34.0,46.318,3.86900,...,NaN,NaN,approved,Laatste inzichten,NaN,NaN,2022-01-01,-4.0,8.0,long_term_gap


In [78]:
"""
================================================================================
FLEXIBILITY VALORIZATION ANALYSIS — DUTCH GRID CAPACITY DATA
================================================================================
Author  : Analytics Engineering — Energy Markets
Version : 1.0
Purpose : Identify postcodes and supply areas with the highest potential for
          energy flexibility programs (load-shifting, feed-in valorization).

BUSINESS RATIONALE
------------------
Energy flexibility is valuable when two conditions align:
  1. The local grid is already stressed (congested), meaning the DSO cannot
     simply build its way out of the problem in the short term.
  2. There is *unmet demand* for capacity — evidenced by queues — meaning
     a flexibility provider can be paid to either reduce consumption or
     inject energy at the right time instead of waiting for a grid upgrade.

A postcode or region that scores high on both dimensions is the ideal target
for flexibility contracts, demand-response programs, and ancillary service
aggregation.
================================================================================
"""

import pandas as pd
import numpy as np

# ── 0. LOAD RAW DATA ──────────────────────────────────────────────────────────

pc6 = pd.read_csv("raw_data/congestie_pc6.csv", sep=";", decimal=",")
vg  = pd.read_csv("raw_data/voedingsgebieden.csv", sep=";", decimal=",")
tg  = pd.read_csv("raw_data/tennetgebieden.csv", sep=";", decimal=",")
tc  = pd.read_csv("raw_data/tennetcongestie.csv", sep=";", decimal=",")
proj = pd.read_csv("raw_data/projecten.csv", sep=";", decimal=",")


# ── 1. RENAME COLUMNS ─────────────────────────────────────────────────────────

pc6 = pc6.rename(columns={
    "postcode":           "postcode",
    "afname":             "consumption_color",   # 0=transparent … 3=red
    "opwek":              "generation_color",
    "voedingsgebied_id":  "supply_area_id",
    "voedingsgebied_naam":"supply_area_name",
    "tennet_id":          "tennet_id",
    "RNB_postcode":       "regional_grid_operator",
})

vg = vg.rename(columns={
    "voedingsgebied_id":                         "supply_area_id",
    "aanwezige_transportcapaciteit_invoeding":   "available_feedin_capacity_mw",
    "aanwezige_transportcapaciteit_afname":      "available_consumption_capacity_mw",
    "benodigde_transportcapaciteit_invoeding":   "required_feedin_capacity_mw",
    "benodigde_transportcapaciteit_afname":      "required_consumption_capacity_mw",
    "unieke_verzoeken_invoeding":                "unique_feedin_requests",
    "unieke_verzoeken_afname":                   "unique_consumption_requests",
    "wachtrij_invoeding":                        "feedin_queue_mw",
    "wachtrij_afname":                           "consumption_queue_mw",
    "RNB":                                       "regional_grid_operator",
    "info":                                      "additional_info",
})

tg = tg.rename(columns={
    "aanwezige_transportcapaciteit_invoeding":   "available_feedin_capacity_mw",
    "aanwezige_transportcapaciteit_afname":      "available_consumption_capacity_mw",
    "benodigde_transportcapaciteit_invoeding":   "required_feedin_capacity_mw",
    "benodigde_transportcapaciteit_afname":      "required_consumption_capacity_mw",
    "unieke_verzoeken_invoeding":                "unique_feedin_requests",
    "unieke_verzoeken_afname":                   "unique_consumption_requests",
    "wachtrij_invoeding":                        "feedin_queue_mw",
    "wachtrij_afname":                           "consumption_queue_mw",
    "congestiegebied":                           "congestion_area",
    "info":                                      "additional_info",
})

tc = tc.rename(columns={
    "tennet_id":             "tennet_id",
    "congestiegebied_afname":"consumption_congestion_area",
    "congestiegebied_opwek": "generation_congestion_area",
    "afname":                "consumption_congestion",
    "opwek":                 "generation_congestion",
})

proj = proj.rename(columns={
    "projectnaam":  "project_name",
    "gebied_id":    "area_id",
    "kwartaal":     "completion_quarter",
    "jaar":         "completion_year",
    "beschrijving": "description",
    "einddatum":    "completion_date",
    "netbeheerder": "grid_operator",
})




In [79]:
# ── 2. ENRICH SUPPLY-AREA TABLE WITH DERIVED METRICS ─────────────────────────
#
# WHY THESE RATIOS?
# -----------------
# The key economic insight is: where a grid is *full* (required ≥ available),
# a flexibility provider acts as a virtual grid upgrade. The DSO is willing to
# pay for this because building hardware takes years. The metrics below quantify
# exactly how "full" a grid is, and how long the queue of frustrated customers
# waiting for capacity already is — both signals of willingness-to-pay.

EPSILON = 1e-9   # prevents division by zero for areas with 0 available capacity


# 2a. CAPACITY UTILISATION RATIOS
# --------------------------------
# Ratio > 1.0  → grid is over-subscribed; congestion is structural.
# Ratio = 1.0  → grid is exactly at the limit; any new connection tips it over.
# Ratio < 1.0  → headroom exists; flexibility is less urgent here.
#
# For FEED-IN: high ratio = solar/wind owners can't export → they need someone
#              to consume or store during generation peaks (load-shift buyers).
# For CONSUMPTION: high ratio = industrial/EV demand cannot be met → load
#                  shedding programs or flexible contracts are very valuable.

vg["feedin_utilisation_ratio"] = (
    vg["required_feedin_capacity_mw"] /
    (vg["available_feedin_capacity_mw"] + EPSILON)
)

vg["consumption_utilisation_ratio"] = (
    vg["required_consumption_capacity_mw"] /
    (vg["available_consumption_capacity_mw"] + EPSILON)
)

# 2b. QUEUE INTENSITY RATIOS
# ---------------------------
# Queue / Available tells us: "For every MW of existing capacity, how many MW
# are waiting in line?"  A ratio of 2.0 means the queue is twice the current
# grid capacity — an extreme signal of unmet demand and willingness to pay.
#
# This ratio also predicts *persistence* of the market: a large queue means
# the opportunity won't disappear overnight even after partial grid upgrades.

vg["feedin_queue_intensity"] = (
    vg["feedin_queue_mw"] /
    (vg["available_feedin_capacity_mw"] + EPSILON)
)

vg["consumption_queue_intensity"] = (
    vg["consumption_queue_mw"] /
    (vg["available_consumption_capacity_mw"] + EPSILON)
)

# 2c. DEMAND-SUPPLY IMBALANCE (absolute gap)
# -------------------------------------------
# Required minus Available in MW.  Positive = congested.
# We keep this in MW because it anchors the economic opportunity:
# 1 MW of flexibility in a congested area can avoid the DSO paying for a
# transformer upgrade (typically €500k–€2M per MVA in NL).

vg["feedin_capacity_gap_mw"] = (
    vg["required_feedin_capacity_mw"] - vg["available_feedin_capacity_mw"]
)

vg["consumption_capacity_gap_mw"] = (
    vg["required_consumption_capacity_mw"] - vg["available_consumption_capacity_mw"]
)

# 2d. COMBINED STRESS INDEX
# --------------------------
# Average of feed-in and consumption utilisation, then log-scaled.
# Log scaling compresses extreme outliers (a ratio of 50x shouldn't dominate
# over a ratio of 10x by a factor of 5 in the final score).
# This gives a single "how stressed is this area?" number per supply area.

vg["grid_stress_index"] = np.log1p(
    (vg["feedin_utilisation_ratio"] + vg["consumption_utilisation_ratio"]) / 2
)

# 2e. TOTAL QUEUE PRESSURE (MW)
# ------------------------------
# Sum of feed-in and consumption queues. This is the raw MW of unmet demand —
# a direct proxy for market size in flexibility contracting.

vg["total_queue_mw"] = vg["feedin_queue_mw"] + vg["consumption_queue_mw"]

# 2f. REQUEST DIVERSITY INDEX
# ----------------------------
# (total unique requests) / (total queue MW)  — higher means many small players.
# Many small players = aggregation opportunity (flexibility aggregators profit
# by bundling small assets).  Low = few large players; direct B2B contracts.

vg["request_diversity_index"] = (
    (vg["unique_feedin_requests"] + vg["unique_consumption_requests"]) /
    (vg["total_queue_mw"] + EPSILON)
)




In [81]:
proj

,Unnamed: 0,project_name,area_id,completion_quarter,completion_year,description,jaar_num,kwartaal_num,einddatum_string,grid_operator,start,eind,tijdschema_beschrijving,project_fase,bron,project_url,project_type
0,NaN,Extra capaciteit op Californië,CALF,NaN,2029,NaN,NaN,NaN,2029,Enexis,NaN,NaN,NaN,scheduled,Laatste inzichten,NaN,NaN
1,NaN,Extra capaciteit op Nederweert,NEDW,NaN,2029,NaN,NaN,NaN,2029,Enexis,NaN,NaN,NaN,scheduled,Laatste inzichten,NaN,NaN
2,NaN,Extra capaciteit op Hengelo Bolderhoek,HGLB,NaN,2028,NaN,NaN,NaN,2028,Enexis,NaN,NaN,NaN,inProgress,Laatste inzichten,NaN,NaN
3,NaN,Extra capaciteit op Tilburg West,TBW,NaN,2029,NaN,NaN,NaN,2029,Enexis,NaN,NaN,NaN,scheduled,Laatste inzichten,NaN,NaN
4,NaN,Extra capaciteit op Ommen Dante,OMD,NaN,2018,NaN,NaN,NaN,2018,Enexis,NaN,NaN,NaN,approved,Laatste inzichten,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
940,NaN,Borssele-Rilland 380 kV verzwaren verbinding,Zeeland,4.0,2037,NaN,NaN,NaN,2037 - Q4,TenneT,2033-01-01 00:00:00,2037-12-31,NaN,approved,Investeringsplan,NaN,NaN
941,NaN,Terneuzen 380 kV realiseren station,Zeeland,4.0,2037,NaN,NaN,NaN,2037 - Q4,TenneT,2034-01-01 00:00:00,2037-12-31,NaN,scheduled,Investeringsplan,NaN,NaN
942,NaN,Borssele-Goes de Poel-Terneuzen 150 kV aanpass...,Zeeland,4.0,2038,NaN,NaN,NaN,2038 - Q4,TenneT,2035-01-01 00:00:00,2038-12-31,NaN,approved,Investeringsplan,NaN,NaN
943,NaN,Terneuzen-Westdorpe 150 kV verzwaren verbinding,Zeeland,4.0,2038,NaN,NaN,NaN,2038 - Q4,TenneT,2035-01-01 00:00:00,2038-12-31,NaN,approved,Investeringsplan,NaN,NaN


In [83]:
print(proj.columns)
print(proj[["completion_date"]].head())

Index(['Unnamed: 0', 'project_name', 'area_id', 'completion_quarter',
       'completion_year', 'description', 'jaar_num', 'kwartaal_num',
       'einddatum_string', 'grid_operator', 'start', 'eind',
       'tijdschema_beschrijving', 'project_fase', 'bron', 'project_url',
       'project_type'],
      dtype='str')


KeyError: "None of [Index(['completion_date'], dtype='str')] are in the [columns]"

In [ ]:
# ── 3. BUILD PLANNED-RELIEF LOOKUP FROM PROJECTS ─────────────────────────────
#
# WHY: If a large upgrade project is already scheduled for an area, the
# congestion window is *temporary* — a flexibility contract signed today may
# be worthless in 18 months. We discount areas with imminent planned relief.
#
# We count active projects per area_id. Areas with many projects scheduled
# will see congestion resolved sooner; areas with zero projects will need
# market-based solutions for the longest time.

proj["correct_column_name"] = pd.to_datetime(
    proj["correct_column_name"], 
    errors="coerce"
)


# Flag projects completing within 2 years as "near-term relief"
NEAR_TERM_HORIZON = pd.Timestamp.today() + pd.DateOffset(years=2)

near_term_projects = (
    proj[proj["completion_date"] <= NEAR_TERM_HORIZON]
    .groupby("area_id")
    .size()
    .reset_index(name="near_term_project_count")
)

all_projects = (
    proj
    .groupby("area_id")
    .size()
    .reset_index(name="total_project_count")
)

project_relief = all_projects.merge(near_term_projects, on="area_id", how="left")
project_relief["near_term_project_count"] = (
    project_relief["near_term_project_count"].fillna(0).astype(int)
)

# Join relief data to supply areas
vg = vg.merge(
    project_relief.rename(columns={"area_id": "supply_area_id"}),
    on="supply_area_id",
    how="left"
)
vg["total_project_count"]    = vg["total_project_count"].fillna(0).astype(int)
vg["near_term_project_count"]= vg["near_term_project_count"].fillna(0).astype(int)

# Relief discount factor: 1.0 = no imminent relief (full score retained)
# Each near-term project shaves 15% off the opportunity score (congestion
# will be partly solved). Floor at 0.1 so no area scores zero just from projects.
vg["relief_discount"] = np.clip(
    1.0 - (vg["near_term_project_count"] * 0.15),
    a_min=0.1, a_max=1.0
)




KeyError: 'correct_column_name'

In [ ]:
# ── 4. COMPOSITE VALORIZATION SCORE (SUPPLY AREA LEVEL) ──────────────────────
#
# WEIGHTING RATIONALE
# --------------------
# w1 — grid_stress_index          (0.30) : Core signal. Structural congestion
#       is the primary prerequisite for any valorization contract.
#
# w2 — consumption_utilisation_ratio (0.25) : Weighted higher than feed-in
#       because *consumption flexibility* (industrial load-shift, EV smart
#       charging) commands premium in evening peak pricing windows (EPEX day-
#       ahead price spikes typically driven by evening demand peaks).
#
# w3 — feedin_utilisation_ratio   (0.15) : Feed-in congestion creates curtail-
#       ment risk for generators — they'll pay for flexibility but the margin
#       is lower than demand-side response.
#
# w4 — feedin_queue_intensity     (0.15) : Queue depth on supply side reveals
#       how many renewable projects are blocked — a large queue means the DSO
#       has committed to paying for solutions.
#
# w5 — consumption_queue_intensity (0.15) : Same logic for demand side. A
#       heavy consumption queue means industrial players are actively seeking
#       connection — high intent signals higher contract probability.
#
# All components are min-max normalised to [0, 1] before weighting so they
# contribute proportionally regardless of absolute MW magnitudes.

WEIGHTS = {
    "grid_stress_index":              0.30,
    "consumption_utilisation_ratio":  0.25,
    "feedin_utilisation_ratio":       0.15,
    "feedin_queue_intensity":         0.15,
    "consumption_queue_intensity":    0.15,
}

def minmax_scale(series: pd.Series) -> pd.Series:
    """Scale a series to [0, 1]. Returns 0.5 for constant series."""
    lo, hi = series.min(), series.max()
    if hi == lo:
        return pd.Series(0.5, index=series.index)
    return (series - lo) / (hi - lo)

# Normalise each component
score_components = pd.DataFrame(index=vg.index)
for col, weight in WEIGHTS.items():
    score_components[f"{col}_norm"] = minmax_scale(vg[col]) * weight

# Raw weighted score
vg["raw_valorization_score"] = score_components.sum(axis=1)

# Apply relief discount: high near-term project count reduces the score
vg["valorization_score"] = vg["raw_valorization_score"] * vg["relief_discount"]

# Final rank (1 = highest opportunity)
vg["valorization_rank"] = vg["valorization_score"].rank(
    ascending=False, method="min"
).astype(int)

# Tier classification for easy filtering
vg["opportunity_tier"] = pd.cut(
    vg["valorization_score"],
    bins=[-np.inf, 0.2, 0.4, 0.6, 0.8, np.inf],
    labels=["Tier 5 — Minimal", "Tier 4 — Low", "Tier 3 — Moderate",
            "Tier 2 — High", "Tier 1 — Priority"],
)


# ── 5. MAP SCORES BACK TO POSTCODE LEVEL (pc6) ────────────────────────────────
#
# pc6 carries congestion color codes (0–3) per postcode.
# We join the supply-area-level scores down to pc6 level so that commercial
# teams can target by postcode (useful for field sales routing, mailing campaigns).

pc6 = pc6.merge(
    vg[[
        "supply_area_id", "valorization_score", "valorization_rank",
        "opportunity_tier", "grid_stress_index",
        "feedin_utilisation_ratio", "consumption_utilisation_ratio",
        "feedin_queue_intensity", "consumption_queue_intensity",
        "total_queue_mw", "feedin_capacity_gap_mw", "consumption_capacity_gap_mw",
        "near_term_project_count", "relief_discount",
    ]],
    on="supply_area_id",
    how="left"
)

# Postcode-level congestion severity (sum of both color codes: max = 6)
# Color codes: -1 = unknown, 0 = ok, 1 = yellow, 2 = orange, 3 = red
# We zero out -1 (no information) so it doesn't distort the severity sum.
pc6["consumption_color_clean"] = pc6["consumption_color"].clip(lower=0)
pc6["generation_color_clean"]  = pc6["generation_color"].clip(lower=0)
pc6["postcode_congestion_severity"] = (
    pc6["consumption_color_clean"] + pc6["generation_color_clean"]
)

# Rank postcodes within their supply area by the congestion severity as a
# tie-breaker: if two postcodes share the same supply-area score, the one
# with the most severe local congestion coloring ranks higher.
pc6 = pc6.sort_values(
    ["valorization_score", "postcode_congestion_severity"],
    ascending=[False, False]
).reset_index(drop=True)

pc6["postcode_rank"] = pc6.index + 1


# ── 6. REGIONAL AGGREGATION & SUMMARY STATISTICS ─────────────────────────────
#
# Aggregate from postcode level up to supply_area / regional_grid_operator.
# Summary statistics help identify not just the *average* opportunity in a
# region, but also *consistency* (low std dev = entire region is congested)
# versus *patchiness* (high std dev = only a few hotspots).

pc6_agg = (
    pc6.groupby(["supply_area_id", "supply_area_name", "regional_grid_operator"])
    .agg(
        postcode_count          = ("postcode", "count"),
        mean_congestion_severity= ("postcode_congestion_severity", "mean"),
        median_congestion_severity=("postcode_congestion_severity", "median"),
        std_congestion_severity = ("postcode_congestion_severity", "std"),
        pct_red_consumption     = ("consumption_color_clean", lambda x: (x == 3).mean()),
        pct_red_generation      = ("generation_color_clean",  lambda x: (x == 3).mean()),
    )
    .reset_index()
)

# Merge back supply-area-level valorization data
regional_summary = pc6_agg.merge(
    vg[[
        "supply_area_id", "valorization_score", "valorization_rank",
        "opportunity_tier", "grid_stress_index",
        "feedin_utilisation_ratio", "consumption_utilisation_ratio",
        "total_queue_mw", "feedin_capacity_gap_mw", "consumption_capacity_gap_mw",
        "near_term_project_count", "relief_discount",
    ]],
    on="supply_area_id",
    how="left"
).sort_values("valorization_rank")


# ── 7. OPERATOR-LEVEL ROLL-UP ─────────────────────────────────────────────────
#
# Useful for strategic account targeting: which DSO operates the most congested
# grid footprint in total? Prioritise relationship-building accordingly.

operator_summary = (
    regional_summary.groupby("regional_grid_operator")
    .agg(
        supply_area_count       = ("supply_area_id", "count"),
        total_postcode_count    = ("postcode_count", "sum"),
        mean_valorization_score = ("valorization_score", "mean"),
        median_valorization_score=("valorization_score", "median"),
        std_valorization_score  = ("valorization_score", "std"),
        total_queue_mw          = ("total_queue_mw", "sum"),
        priority_areas          = ("opportunity_tier",
                                   lambda x: (x == "Tier 1 — Priority").sum()),
    )
    .reset_index()
    .sort_values("mean_valorization_score", ascending=False)
)


# ── 8. FINAL EXPORT ───────────────────────────────────────────────────────────

# Ranked postcode list (primary commercial deliverable)
pc6_export_cols = [
    "postcode_rank", "postcode", "supply_area_id", "supply_area_name",
    "regional_grid_operator", "tennet_id",
    "consumption_color", "generation_color", "postcode_congestion_severity",
    "valorization_score", "opportunity_tier",
    "feedin_utilisation_ratio", "consumption_utilisation_ratio",
    "feedin_queue_intensity", "consumption_queue_intensity",
    "total_queue_mw", "near_term_project_count",
]

ranked_postcodes     = pc6[pc6_export_cols].copy()
ranked_supply_areas  = regional_summary.copy()
ranked_operators     = operator_summary.copy()

print("=" * 72)
print("TOP 20 POSTCODES BY VALORIZATION SCORE")
print("=" * 72)
print(ranked_postcodes.head(20).to_string(index=False))

print("\n" + "=" * 72)
print("TOP 15 SUPPLY AREAS BY VALORIZATION SCORE")
print("=" * 72)
print(ranked_supply_areas.head(15)[[
    "valorization_rank", "supply_area_name", "regional_grid_operator",
    "valorization_score", "opportunity_tier",
    "feedin_utilisation_ratio", "consumption_utilisation_ratio",
    "total_queue_mw", "near_term_project_count",
]].to_string(index=False))

print("\n" + "=" * 72)
print("OPERATOR ROLL-UP")
print("=" * 72)
print(ranked_operators.to_string(index=False))

# Save outputs
ranked_postcodes.to_csv("outputs/ranked_postcodes.csv", index=False)
ranked_supply_areas.to_csv("outputs/ranked_supply_areas.csv", index=False)
ranked_operators.to_csv("outputs/ranked_operators.csv", index=False)

print("\n✓ Outputs saved to outputs/")